In [2]:
import argos
from argos import ArgosDataset
import pandas as pd
import numpy as np
import re

## Importação de dados

In [3]:
metadata = pd.read_csv("../data/processed/metadata_filtrada.csv")
metadata.head()

,Species,Source,Date,Location,sample,BioSample,Estado,Região,source_type,WHO_Priority
0,Leptospira interrogans,Homo sapiens,NaN,Salvador,GCA_000216055.2,SAMN00254327,BA,Nordeste,Other,Other
1,Escherichia coli,NaN,1990.0,Brazil,GCA_000316425.1,SAMN01041333,NaN,NaN,Unknown,Critical
2,Vibrio cholerae,patient with cholera-like diarrhea,1991.0,NaN,GCA_000223095.2,SAMN02470783,NaN,NaN,Gastrointestinal,Other
3,Acinetobacter bereziniae,Rectal Swab,2019.0,Brazil,GCA_036761135.1,SAMN39408214,NaN,NaN,Gastrointestinal,Other
4,Acinetobacter bereziniae,Endotracheal aspirate,2014.0,"Londrina, PR",GCA_003670255.1,SAMN09907131,PR,Sul,Respiratory,Other


In [4]:
ids_finais = metadata["sample"].tolist()
dataset = ArgosDataset.from_parquet("../data/argos_project/parquet_data", samples=ids_finais)
dataset.resistome.head()

,ORF_ID,Cut_Off,Pass_Bitscore,Best_Hit_Bitscore,Best_Identities,drug_class,mechanism,amr_gene_family,Predicted_Protein,CARD_Protein_Sequence,...,gene,aro_accession,model_type,gene_clean,seq_name,start,end,strand,length_nt,contig_type
0,ABSC01000258.1_1 # 45 # 395 # 1 # ID=83_1;part...,Strict,100,115.931,42.98,{phosphonic acid antibiotic},{antibiotic inactivation},{fosfomycin thiol transferase},MLNTLFDAKEIYDSQQKKFSLYPEKFFLVKDLWIAVMQNSSNKLPK...,MKGISHITFIVRDLNRMAALLCEGLGAREVYDSSNQNFSLSREKFF...,...,FosI,ARO:3007370,protein homolog model,FosI,ABSC01000258.1,45,395,1,351,chromosome
1,ABSC01000180.1_7 # 6357 # 7019 # -1 # ID=161_7...,Strict,50,220.705,57.14,{glycopeptide antibiotic},{antibiotic target alteration},"{glycopeptide resistance gene cluster, vanY}",MKRSYKTVAVILLIVLLASIGLFFRKTPQKIQVCGEERDSWNLLLV...,MEKSNYHSNVNHHKRHMKQSGEKRAFLWAFIISFTVCTLFLGWRLV...,...,vanY gene in vanB cluster,ARO:3002956,protein homolog model,vanY_in_vanB_cl,ABSC01000180.1,6357,7019,-1,663,chromosome
2,ABSC01000175.1_1 # 81 # 1520 # -1 # ID=166_1;p...,Strict,900,966.066,99.79,{aminoglycoside antibiotic},{antibiotic inactivation},{aminoglycoside bifunctional resistance protein},MNIVENEICIRTLIDDDFPLMLKWLTDERVLEFYDGRDKKYTLESL...,MNIVENEICIRTLIDDDFPLMLKWLTDERVLEFYGGRDKKYTLESL...,...,AAC(6')-Ie-APH(2'')-Ia bifunctional protein,ARO:3002597,protein homolog model,AAC6_Ie_APH2_Ia,ABSC01000175.1,81,1520,-1,1440,chromosome
3,ABSC01000090.1_4 # 3146 # 4876 # 1 # ID=251_4;...,Strict,960,989.949,82.81,"{fluoroquinolone antibiotic, macrolide antibio...",{antibiotic efflux},{ATP-binding cassette (ABC) antibiotic efflux ...,MKLMWHYTMRYKKFLFFNFICVFGFILIELGLPTILARMIDVGIRN...,MKLMWRYTMRYKKLLFADFICVFGFILIELGLPTILARMIDKGIIP...,...,efrA,ARO:3003948,protein homolog model,efrA,ABSC01000090.1,3146,4876,1,1731,chromosome
4,ABSC01000050.1_4 # 2667 # 3404 # 1 # ID=291_4;...,Strict,400,489.574,97.96,"{lincosamide antibiotic, macrolide antibiotic,...",{antibiotic target alteration},{Erm 23S ribosomal RNA methyltransferase},MNKNIKYSQNFLTSEKVLNQIIKQLNLKETDTVYEIGTGKGHLTTK...,MNKNIKYSQNFLTSEKVLNQIIKQLNLKETDTVYEIGTGKGHLTTK...,...,ErmB,ARO:3000375,protein homolog model,ErmB,ABSC01000050.1,2667,3404,1,738,plasmid


In [5]:
vfdb = pd.read_csv('../data/raw/vfdb_all_hits.csv')
vfdb.head()

,#FILE,SEQUENCE,START,END,STRAND,GENE,COVERAGE,COVERAGE_MAP,GAPS,%COVERAGE,%IDENTITY,DATABASE,ACCESSION,PRODUCT,RESISTANCE,sample,categoria
0,../genomes//GCA_052400065.1_ASM5240006v1_genom...,JBQTZX010000001.1,6381,7271,-,inlC,1-891/891,===============,0/0,100.0,97.19,vfdb,NP_465311,(inlC) internalin C [InlC (VF0438) - Immune mo...,NaN,GCA_052400065.1_ASM5240006v1_genomic,Immune modulation
1,../genomes//GCA_052400065.1_ASM5240006v1_genom...,JBQTZX010000001.1,50025,51737,+,fbpA,1-1713/1713,===============,0/0,100.0,91.71,vfdb,NP_465354,(fbpA) fibronectin-binding protein [FbpA (VF03...,NaN,GCA_052400065.1_ASM5240006v1_genomic,Adherence
2,../genomes//GCA_052400065.1_ASM5240006v1_genom...,JBQTZX010000001.1,66308,66772,-,lspA,1-465/465,===============,0/0,100.0,97.42,vfdb,NP_465369,(lspA) signal peptidase II [Lsp (VF0351) - Pos...,NaN,GCA_052400065.1_ASM5240006v1_genomic,Post-translational modification
3,../genomes//GCA_052400065.1_ASM5240006v1_genom...,JBQTZX010000001.1,69697,70629,-,lpeA,1-933/933,===============,0/0,100.0,96.68,vfdb,NP_465372,(lpeA) lipoprotein promoting cell invasion [Lp...,NaN,GCA_052400065.1_ASM5240006v1_genomic,Invasion
4,../genomes//GCA_052400065.1_ASM5240006v1_genom...,JBQTZX010000001.1,301597,302574,-,bsh,1-978/978,===============,0/0,100.0,97.75,vfdb,NP_465591,(bsh) bile salt hydrolase [BSH (VF0350) - Stre...,NaN,GCA_052400065.1_ASM5240006v1_genomic,Stress survival


In [6]:
gc = pd.read_csv("../data/raw/gc_stats.txt", sep=r"\s+", thousands=",")
gc.head()

,file,format,type,num_seqs,sum_len,min_len,avg_len,max_len,Q1,Q2,Q3,sum_gap,N50,N50_num,Q20(%),Q30(%),AvgQual,GC(%),sum_n
0,genomes_contig_type/GCA_000172875.1_ASM17287v1...,FASTA,DNA,293,2486720,216,8487.1,150593,996.0,2832.0,9453.0,0,22620,28,0,0,0,37.94,0.0
1,genomes_contig_type/GCA_000172875.1_ASM17287v1...,FASTA,DNA,34,364754,2531,10728.1,36405,4772.0,9614.5,12656.0,0,12656,9,0,0,0,36.24,0.0
2,genomes_contig_type/GCA_000188755.2_ASM18875v2...,FASTA,DNA,116,4411386,277,38029.2,222066,2212.5,19067.5,51921.0,0,91542,17,0,0,0,50.74,4.0
3,genomes_contig_type/GCA_000188755.2_ASM18875v2...,FASTA,DNA,36,744317,954,20675.5,90331,7710.0,13332.5,26628.5,0,29539,7,0,0,0,49.15,1.0
4,genomes_contig_type/GCA_000188815.2_ASM18881v2...,FASTA,DNA,46,4960945,234,107846.6,730778,683.0,41411.5,198155.0,0,249753,7,0,0,0,50.84,0.0


In [7]:
# NCBI AMRFinderPlus Reference Gene Catalog -- fonte estruturada usada mais
# abaixo pra classificar cada beta-lactamase por classe de Ambler (A/B/C/D),
# em vez de tentar adivinhar por sigla. Baixado em https://www.ncbi.nlm.nih.gov/pathogens/refgene/
ref = pd.read_csv("../data/raw/refgenes.tsv", sep="\t")
ref.columns = [c.lstrip("#") for c in ref.columns]  # normaliza "#Allele" -> "Allele"
print(ref.columns.tolist())  # confere os nomes reais das colunas antes de seguir
ref.head()

['Allele', 'Gene family', 'Product name', 'Scope', 'Type', 'Subtype', 'Class', 'Subclass', 'RefSeq protein', 'RefSeq nucleotide', 'GenBank protein', 'GenBank nucleotide', 'Curated RefSeq start', 'Links']


,Allele,Gene family,Product name,Scope,Type,Subtype,Class,Subclass,RefSeq protein,RefSeq nucleotide,GenBank protein,GenBank nucleotide,Curated RefSeq start,Links
0,16S_A1055G,16S,16S ribosomal RNA,core,AMR,POINT,TETRACYCLINE,TETRACYCLINE,NaN,NC_000913.3,NaN,U00096.3,NaN,0
1,16S_A1408G,16S,16S ribosomal RNA,core,AMR,POINT,AMINOGLYCOSIDE,GENTAMICIN C/NEOMYCIN/PAROMOMYCIN,NaN,NC_000913.3,NaN,U00096.3,NaN,0
2,16S_A1499G,16S,16S ribosomal RNA,core,AMR,POINT,EDEINE,EDEINE,NaN,NC_000913.3,NaN,U00096.3,NaN,0
3,16S_A1519C,16S,16S ribosomal RNA,core,AMR,POINT,AMINOGLYCOSIDE,KASUGAMYCIN,NaN,NC_000913.3,NaN,U00096.3,NaN,0
4,16S_A1519G,16S,16S ribosomal RNA,core,AMR,POINT,AMINOGLYCOSIDE,KASUGAMYCIN,NaN,NC_000913.3,NaN,U00096.3,NaN,0


## Ajustando dados importados

In [8]:
# Reindexando o `metadata` pra eliminar erro futuro
metadata = metadata.set_index("sample")
metadata.head(n=3)

,Species,Source,Date,Location,BioSample,Estado,Região,source_type,WHO_Priority
sample,,,,,,,,,
GCA_000216055.2,Leptospira interrogans,Homo sapiens,NaN,Salvador,SAMN00254327,BA,Nordeste,Other,Other
GCA_000316425.1,Escherichia coli,NaN,1990.0,Brazil,SAMN01041333,NaN,NaN,Unknown,Critical
GCA_000223095.2,Vibrio cholerae,patient with cholera-like diarrhea,1991.0,NaN,SAMN02470783,NaN,NaN,Gastrointestinal,Other


### Dados do seqkit stats

In [9]:
# Conteúdo GC em contigs cromossomais vs plasmidiais
gc["sample_id"] = gc["file"].str.extract(r"(GC[AF]_\d+\.\d+)")
gc["contig_type"] = gc["file"].str.extract(r"_(chromosome|plasmid)\.fna$")

gc_wide = gc.pivot(index="sample_id", columns="contig_type", values="GC(%)")
gc_wide.columns = ["gc_cromossomal", "gc_plasmidial"]
gc_wide["gc_diff_plasmid_cromossomo"] = gc_wide["gc_plasmidial"] - gc_wide["gc_cromossomal"]
gc_wide = gc_wide.reindex(metadata.index)

gc_wide.head()

,gc_cromossomal,gc_plasmidial,gc_diff_plasmid_cromossomo
sample,,,
GCA_000216055.2,34.95,35.13,0.18
GCA_000316425.1,50.57,47.65,-2.92
GCA_000223095.2,47.69,44.64,-3.05
GCA_036761135.1,37.79,38.10,0.31
GCA_003670255.1,37.95,38.85,0.90


### Tamanho cromossomal (Mb) -- pra normalizar densidade de gene

Usado mais abaixo (`densidade_genes_cromossomal_por_mb`, `n_proviruses_por_mb`) pra comparar
genomas de tamanho diferente na mesma escala -- só a parte CROMOSSOMAL do genoma, não o
assembly inteiro, senão a presença/ausência de plasmídeo distorceria a normalização.

In [10]:
genoma_mb_cromo = (
    dataset.contigs[dataset.contigs["contig_type"] == "chromosome"]
    .groupby("sample_id")["length_nt"].sum() / 1e6
).reindex(metadata.index)

genoma_mb_cromo.head()

sample
GCA_000216055.2    0.0
GCA_000316425.1    0.0
GCA_000223095.2    0.0
GCA_036761135.1    0.0
GCA_003670255.1    0.0
Name: length_nt, dtype: float64

### VFDB -- sample_id e categoria (a partir do PRODUCT)

In [11]:
vfdb["sample_id"] = vfdb["sample"].str.extract(r"(GC[AF]_\d+\.\d+)")

# extrai a categoria (Adherence, Biofilm, Nutritional/Metabolic factor...) do
# campo PRODUCT do abricate -- ancora especificamente em "(VFCxxxx)", que só
# aparece 1x por linha, junto da categoria -- robusto mesmo com parêntese
# sobrando no meio do nome do gene (ex: "Cu(+)/Ag(+)")
def extrair_categoria_vfdb(product):
    m = re.search(r"-\s*([^()]+?)\s*\(VFC\d+\)", str(product))
    return m.group(1).strip() if m else None

vfdb["categoria"] = vfdb["PRODUCT"].apply(extrair_categoria_vfdb)

print("linhas sem categoria extraída:", vfdb["categoria"].isna().sum(), "de", len(vfdb))
vfdb[["sample_id", "PRODUCT", "categoria"]].head()

linhas sem categoria extraída: 0 de 571058


,sample_id,PRODUCT,categoria
0,GCA_052400065.1,(inlC) internalin C [InlC (VF0438) - Immune mo...,Immune modulation
1,GCA_052400065.1,(fbpA) fibronectin-binding protein [FbpA (VF03...,Adherence
2,GCA_052400065.1,(lspA) signal peptidase II [Lsp (VF0351) - Pos...,Post-translational modification
3,GCA_052400065.1,(lpeA) lipoprotein promoting cell invasion [Lp...,Invasion
4,GCA_052400065.1,(bsh) bile salt hydrolase [BSH (VF0350) - Stre...,Stress survival


## Fazendo classificação de beta-lactamase

A especificidade de substrato pode ser relativamente restrita ou ampla, incluindo cefalosporinas de amplo espectro e carbapenêmicos.
- **classe A** (conhecidas principalmente como penicilinases) tendem a hidrolisar penicilinas em detrimento das cefalosporinas como substratos, embora muitas variantes possam hidrolisar significativamente cefalosporinas de amplo espectro e carbapenêmicos
- **classe B** (metalo-β-lactamases) tipicamente apresentam uma especificidade de substrato extremamente ampla, incluindo todos os β-lactâmicos, exceto os monobactâmicos (aztreonam)
- **classe C** (cefalosporinases) tendem a preferir as cefalosporinas como substratos
- **classe D** (oxacilinases) apresentam uma preferência excepcionalmente alta por oxacilina e penicilinas relacionadas

Classe de Ambler é **estrutura**, não é o mesmo que "é carbapenemase" -- a maioria dos genes
classe A/D é penicilinase/ESBL estreita, só subfamílias específicas carbapenemizam de verdade
(`KPC`, `OXA-48-like`, etc). Por isso os dois eixos ficam separados: `ambler_class` (estrutura,
vem do catálogo do NCBI) e `tem_carbapenem`/`carbapenemase_confirmada` (função, vem direto do
`drug_class` de cada hit -- sem precisar decorar qual subfamília OXA carbapenemiza).

In [12]:
def extrai_classe_ambler(product_name):
    if pd.isna(product_name):
        return None
    s = str(product_name)

    m = re.search(r"class ([ABCD]) beta-lactamase", s, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # classe B não segue o padrão "class X" -- vem como "subclass B1/B2/B3
    # metallo-beta-lactamase" -- guarda a subclasse (B1/B2/B3), não só "B"
    m = re.search(r"subclass (B[123]) metallo-beta-lactamase", s, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # fallback -- metalo-beta-lactamase sem subclasse explícita no nome
    if re.search(r"metallo-beta-lactamase", s, re.IGNORECASE):
        return "B"

    return None

ref["ambler_class"] = ref["Product name"].apply(extrai_classe_ambler)
ref["ambler_class"].value_counts()

ambler_class
A     2247
C     1784
D     1442
B1     472
B3     151
B2      26
B        4
Name: count, dtype: int64

In [13]:
ref_bl = ref.dropna(subset=["ambler_class", "Gene family"]).copy()
ref_bl["familia"] = (
    ref_bl["Gene family"]
    .str.strip()
    .str.replace(r"^bla", "", regex=True, flags=re.IGNORECASE)
)

familias_por_classe = ref_bl.groupby("ambler_class")["familia"].apply(lambda s: sorted(s.unique()))

ambler_A = familias_por_classe.get("A", [])
ambler_B1 = familias_por_classe.get("B1", [])
ambler_B2 = familias_por_classe.get("B2", [])
ambler_B3 = familias_por_classe.get("B3", [])
ambler_B_generico = familias_por_classe.get("B", [])  # sem subclasse especificada
ambler_C = familias_por_classe.get("C", [])
ambler_D = familias_por_classe.get("D", [])

for classe, lista in familias_por_classe.items():
    print(f"{classe}: {len(lista)} famílias")

A: 129 famílias
B: 4 famílias
B1: 54 famílias
B2: 5 famílias
B3: 46 famílias
C: 52 famílias
D: 10 famílias


In [14]:
# EXCECOES_MANUAIS: famílias do seu resistoma que não bateram contra o
# catálogo do NCBI e foram checadas manualmente contra a literatura
EXCECOES_MANUAIS = {
    "class a mycobacterium tuberculosis bla beta-lactamase": "A",
    "mor beta-lactamase": "C",
    "blaa beta-lactamase": "A",
}

def extrai_stem(nome_card):
    m = re.search(r"([\w\-/]+)\s+beta-lactamase", nome_card, re.IGNORECASE)
    stem = m.group(1) if m else nome_card
    return re.sub(r"-(like|type)$", "", stem, flags=re.IGNORECASE)

def bate(stem, familia):
    stem, familia = stem.lower(), familia.lower()
    if len(familia) < 2:   # descarta '' e siglas de 1 letra -- casam com qualquer string
        return False
    return stem == familia or stem.startswith(familia + "-") or familia.startswith(stem + "-")

def classificar_ambler(nome_card):
    if nome_card.lower() in EXCECOES_MANUAIS:
        return EXCECOES_MANUAIS[nome_card.lower()]
    stem = extrai_stem(nome_card)
    for classe, lista in [("A", ambler_A), ("B1", ambler_B1), ("B2", ambler_B2),
                            ("B3", ambler_B3), ("B", ambler_B_generico),
                            ("C", ambler_C), ("D", ambler_D)]:
        if any(bate(stem, f) for f in lista):
            return classe
    return None

### Aplicando no resistoma inteiro

Em `dataset.resistome` inteiro (não só num subconjunto) -- as features finais precisam
existir pra todo genoma do dataset.

In [15]:
# um hit pode ter mais de 1 valor em amr_gene_family (ex: "OXA beta-lactamase;
# OXA-51-like beta-lactamase" juntos) -- pega a primeira classificação válida
# entre os valores do set
dataset.resistome["ambler_class"] = dataset.resistome["amr_gene_family"].apply(
    lambda s: next((classificar_ambler(f) for f in s if classificar_ambler(f)), None)
)

def tem_carbapenem(drug_class_set):
    return any("carbapenem" in dc.lower() for dc in drug_class_set)

dataset.resistome["tem_carbapenem"] = dataset.resistome["drug_class"].apply(tem_carbapenem)

def tem_cefalosporina(drug_class_set):
    return any("cephalosporin" in dc.lower() for dc in drug_class_set)

dataset.resistome["tem_cefalosporina"] = dataset.resistome["drug_class"].apply(tem_cefalosporina)

# carbapenemase confirmada = tem classe de Ambler estrutural E drug_class
# confirma atividade em carbapenêmico -- os dois juntos, não um só. Sem isso
# "n_class_D_oxa" misturaria OXA-1-like (não carbapenemiza) com OXA-48-like
# (carbapenemiza) na mesma contagem.
dataset.resistome["carbapenemase_confirmada"] = (
    dataset.resistome["ambler_class"].isin(["A", "B", "B1", "B2", "B3", "D"])
    & dataset.resistome["tem_carbapenem"]
)

dataset.resistome[["gene_clean", "drug_class", "contig_type", "ambler_class",
                    "tem_carbapenem", "carbapenemase_confirmada"]].head(10)

,gene_clean,drug_class,contig_type,ambler_class,tem_carbapenem,carbapenemase_confirmada
0,FosI,{phosphonic acid antibiotic},chromosome,None,False,False
1,vanY_in_vanB_cl,{glycopeptide antibiotic},chromosome,None,False,False
2,AAC6_Ie_APH2_Ia,{aminoglycoside antibiotic},chromosome,None,False,False
3,efrA,"{fluoroquinolone antibiotic, macrolide antibio...",chromosome,None,False,False
4,ErmB,"{lincosamide antibiotic, macrolide antibiotic,...",plasmid,None,False,False
5,vanR_in_vanA_cl,{glycopeptide antibiotic},plasmid,None,False,False
6,vanS_in_vanA_cl,{glycopeptide antibiotic},plasmid,None,False,False
7,vanH_in_vanA_cl,{glycopeptide antibiotic},plasmid,None,False,False
8,vanA,{glycopeptide antibiotic},plasmid,None,False,False
9,vanX_in_vanA_cl,{glycopeptide antibiotic},plasmid,None,False,False


### Efflux -- mecanismo oficial do CARD + cross-check com catálogo do NCBI

In [16]:
# "antibiotic efflux" e "reduced permeability to antibiotic" -- os dois
# mecanismos que o próprio CARD/RGI atribui a esse tipo de resistência
# (campo `mechanism`, oficial, não inventado por nós)
mecanismos_efflux = {
    "antibiotic efflux",
    "reduced permeability to antibiotic",
}

efflux_ish = dataset.resistome[
    dataset.resistome["mechanism"].apply(lambda x: bool(x & mecanismos_efflux))
]

familias_unicas_efflux = set().union(*efflux_ish["amr_gene_family"]) if len(efflux_ish) else set()
familias_unicas_efflux

{'ATP-binding cassette (ABC) antibiotic efflux pump',
 'General Bacterial Porin with reduced permeability to beta-lactams',
 'General Bacterial Porin with reduced permeability to peptide antibiotics',
 'Intrinsic peptide antibiotic resistant Lps',
 'Outer Membrane Porin (Opr)',
 'Serine/threonine kinases',
 'daptomycin resistant liaR',
 'daptomycin resistant liaS',
 'kdpDE',
 'major facilitator superfamily (MFS) antibiotic efflux pump',
 'metal transporters with antibiotic efflux',
 'multidrug and toxic compound extrusion (MATE) transporter',
 'pmr phosphoethanolamine transferase',
 'resistance-nodulation-cell division (RND) antibiotic efflux pump',
 'small multidrug resistance (SMR) antibiotic efflux pump',
 'transmembrane protein conferring colistin resistance'}

In [17]:
# cross-check contra o catálogo oficial do NCBI (mesma fonte usada pro
# beta-lactamase) -- SÓ INFORMATIVO, não filtra feature nenhuma. Testamos
# (tanto por família quanto por gene individual) e a nomenclatura de sistema
# ("RND antibiotic efflux pump") vs. gene específico ("AcrB" vs "AcrAB", nome
# de subunidade vs. operon) não bate de forma confiável entre as duas bases --
# não é sinal de família não-reconhecida, é convenção de nome diferente.
# `mechanism` já é campo oficial do próprio CARD/RGI, usamos ele direto.
ref_efflux = ref[ref["Product name"].str.contains("efflux", case=False, na=False)].copy()
familias_efflux_oficiais = sorted(ref_efflux["Gene family"].dropna().str.strip().unique())

nao_reconhecidas = [
    f for f in familias_unicas_efflux
    if not any(fo.lower() in f.lower() for fo in familias_efflux_oficiais)
]
print(f"{len(familias_efflux_oficiais)} famílias no catálogo oficial do NCBI (granularidade de gene)")
print(f"{len(nao_reconhecidas)} famílias de efluxo no seu resistoma sem match direto (granularidade de sistema -- esperado)")

193 famílias no catálogo oficial do NCBI (granularidade de gene)
15 famílias de efluxo no seu resistoma sem match direto (granularidade de sistema -- esperado)


## Fazendo as `features` (por genoma)

### Matrizes por compartimento

In [18]:
def matriz_por_contig_type(resistome, contig_type) -> pd.DataFrame:
    hits = resistome[resistome["contig_type"] == contig_type]
    matriz = pd.crosstab(hits["sample_id"], hits["gene_clean"]).clip(upper=1)
    return matriz.reindex(metadata.index, fill_value=0)  # pra genoma sem hit == 0

matriz_cromossomal = matriz_por_contig_type(dataset.resistome, "chromosome")
matriz_plasmidial = matriz_por_contig_type(dataset.resistome, "plasmid")

matriz_cromossomal.head()

gene_clean,AAC(2')-IIa,AAC(2')-Ia,AAC(2')-Ic,AAC(3)-IIc,AAC(3)-IId,AAC(3)-IIe,AAC(3)-IIg,AAC(3)-IVa,AAC(3)-Ia,AAC(3)-VIIa,...,vanX_in_vanP_cl,vanY_in_vanA_cl,vanY_in_vanB_cl,vanY_in_vanD_cl,vanY_in_vanF_cl,vanY_in_vanG_cl,vanY_in_vanM_cl,vanZ_in_vanA_cl,vatD,vatF
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
GCA_000316425.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_000223095.2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_036761135.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_003670255.1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### n_genes e mdr_score

In [19]:
n_genes_cromossomal = matriz_cromossomal.sum(axis=1).rename("n_genes_cromossomal")
n_genes_plasmidial = matriz_plasmidial.sum(axis=1).rename("n_genes_plasmidial")

n_genes_plasmidial.head()

sample
GCA_000216055.2    0
GCA_000316425.1    1
GCA_000223095.2    0
GCA_036761135.1    6
GCA_003670255.1    7
Name: n_genes_plasmidial, dtype: int64

In [20]:
mdr_score_cromossomal = argos.groupby_genes(
    matriz_cromossomal, dataset.annotations, by="drug_class_compacted", agg="max"
).sum(axis=1).rename("mdr_score_cromossomal")

mdr_score_cromossomal.head()

sample
GCA_000216055.2     1
GCA_000316425.1    13
GCA_000223095.2     7
GCA_036761135.1    10
GCA_003670255.1     8
Name: mdr_score_cromossomal, dtype: int64

### Densidade cromossomal por Mb e % de genes de efluxo

In [21]:
densidade_genes_cromossomal_por_mb = (
    n_genes_cromossomal / genoma_mb_cromo
).rename("densidade_genes_cromossomal_por_mb")

densidade_genes_cromossomal_por_mb.head()

sample
GCA_000216055.2    inf
GCA_000316425.1    inf
GCA_000223095.2    inf
GCA_036761135.1    inf
GCA_003670255.1    inf
Name: densidade_genes_cromossomal_por_mb, dtype: float64

In [22]:
def n_genes_efflux_por_contig(resistome, contig_type):
    hits = resistome[
        (resistome["contig_type"] == contig_type)
        & resistome["mechanism"].apply(lambda x: bool(x & mecanismos_efflux))
    ]
    contagem = hits.groupby("sample_id")["gene_clean"].nunique()
    return contagem.reindex(metadata.index, fill_value=0)

n_efflux_cromossomal = n_genes_efflux_por_contig(dataset.resistome, "chromosome")

pct_genes_cromossomal_efflux = (
    100 * n_efflux_cromossomal / n_genes_cromossomal
).replace([np.inf, -np.inf], np.nan).rename("pct_genes_cromossomal_efflux")

pct_genes_cromossomal_efflux.head()

sample
GCA_000216055.2     0.000000
GCA_000316425.1    82.978723
GCA_000223095.2    33.333333
GCA_036761135.1    50.000000
GCA_003670255.1    71.428571
Name: pct_genes_cromossomal_efflux, dtype: float64

### Classes de Ambler, ESBL, efluxo e co-resistências críticas -- por genoma, cromossomal x plasmidial

Contagem de **genes distintos** (`n_genes_por_condicao`) pras condições numéricas, e presença/ausência
(`has_condicao_por_contig`) pras binárias -- as duas sempre separadas por compartimento, pra não
misturar resistoma intrínseco (cromossomo) com adquirido (plasmídeo) na mesma coluna.

In [23]:
def n_genes_por_condicao(resistome, contig_type, mask):
    hits = resistome[(resistome["contig_type"] == contig_type) & mask]
    contagem = hits.groupby("sample_id")["gene_clean"].size()
    return contagem.reindex(metadata.index, fill_value=0)

def has_condicao_por_contig(resistome, contig_type, mask):
    hits = resistome[(resistome["contig_type"] == contig_type) & mask]
    contagem = hits.groupby("sample_id").size()
    return contagem.reindex(metadata.index, fill_value=0).gt(0).astype(int)

In [24]:
r = dataset.resistome

mask_A_carbapenemase = (r["ambler_class"] == "A") & r["carbapenemase_confirmada"]
mask_B_mbl = r["ambler_class"].isin(["B", "B1", "B2", "B3"])
mask_C_ampc = r["ambler_class"] == "C"
# só a fração carbapenemase de classe D (OXA-48-like, OXA-23-like...) -- não
# a classe D inteira, que incluiria OXA-1-like/OXA-9-like (penicilinase estreita)
mask_D_oxa_carbapenemase = (r["ambler_class"] == "D") & r["carbapenemase_confirmada"]
# ESBL = classe A + hidrolisa cefalosporina (via drug_class, por hit) + NÃO é
# carbapenemase. drug_class vem do modelo ARO específico do hit, então já
# distingue TEM-1 estreito de TEM-ESBL sem precisar de lista de alelo curada
mask_esbl = (r["ambler_class"] == "A") & r["tem_cefalosporina"] & ~r["carbapenemase_confirmada"]
mask_efflux = r["mechanism"].apply(lambda x: bool(x & mecanismos_efflux))
mask_mcr = r["amr_gene_family"].apply(lambda s: any("MCR phosphoethanolamine" in f for f in s))
# armA/rmtB -- pan-resistência a aminoglicosídeo por metilação do 16S rRNA,
# categoria própria no CARD, separada de AAC/APH/ANT comuns
mask_aminoglicosideo_panres = r["amr_gene_family"].apply(lambda s: any("16S rRNA methyltransferase" in f for f in s))

In [25]:
#def n_genes_por_condicao(resistome, contig_type, mask):
#    hits = resistome[(resistome["contig_type"] == contig_type) & mask]
 #   contagem = hits.groupby("sample_id")["gene_clean"].nunique()
  #  return contagem.reindex(metadata.index, fill_value=0)


hits = r[(r["contig_type"] == "plasmid") & mask_A_carbapenemase]

contagem = hits.groupby('sample_id')["gene_clean"].size()

contagem = contagem.reindex(metadata.index, fill_value=0)

contagem.value_counts()

gene_clean
0    6134
1     744
2     159
4       1
Name: count, dtype: int64

In [26]:
n_class_A_carbapenemase_cromossomal = n_genes_por_condicao(r, "chromosome", mask_A_carbapenemase).rename("n_class_A_carbapenemase_cromossomal")
n_class_A_carbapenemase_plasmidial = n_genes_por_condicao(r, "plasmid", mask_A_carbapenemase).rename("n_class_A_carbapenemase_plasmidial")

n_class_B_mbl_cromossomal = n_genes_por_condicao(r, "chromosome", mask_B_mbl).rename("n_class_B_mbl_cromossomal")
n_class_B_mbl_plasmidial = n_genes_por_condicao(r, "plasmid", mask_B_mbl).rename("n_class_B_mbl_plasmidial")

n_class_C_ampc_cromossomal = n_genes_por_condicao(r, "chromosome", mask_C_ampc).rename("n_class_C_ampc_cromossomal")
n_class_C_ampc_plasmidial = n_genes_por_condicao(r, "plasmid", mask_C_ampc).rename("n_class_C_ampc_plasmidial")

n_class_D_oxa_cromossomal = n_genes_por_condicao(r, "chromosome", mask_D_oxa_carbapenemase).rename("n_class_D_oxa_cromossomal")
n_class_D_oxa_plasmidial = n_genes_por_condicao(r, "plasmid", mask_D_oxa_carbapenemase).rename("n_class_D_oxa_plasmidial")

n_esbl_cromossomal = n_genes_por_condicao(r, "chromosome", mask_esbl).rename("n_esbl_cromossomal")
n_esbl_plasmidial = n_genes_por_condicao(r, "plasmid", mask_esbl).rename("n_esbl_plasmidial")

n_genes_efflux_cromossomal = n_genes_por_condicao(r, "chromosome", mask_efflux).rename("n_genes_efflux_cromossomal")
n_genes_efflux_plasmidial = n_genes_por_condicao(r, "plasmid", mask_efflux).rename("n_genes_efflux_plasmidial")

has_mcr_colistin_cromossomal = has_condicao_por_contig(r, "chromosome", mask_mcr).rename("has_mcr_colistin_cromossomal")
has_mcr_colistin_plasmidial = has_condicao_por_contig(r, "plasmid", mask_mcr).rename("has_mcr_colistin_plasmidial")

has_aminoglycoside_panres_cromossomal = has_condicao_por_contig(r, "chromosome", mask_aminoglicosideo_panres).rename("has_aminoglycoside_panres_cromossomal")
has_aminoglycoside_panres_plasmidial = has_condicao_por_contig(r, "plasmid", mask_aminoglicosideo_panres).rename("has_aminoglycoside_panres_plasmidial")

pd.concat([
    n_class_A_carbapenemase_plasmidial, n_class_B_mbl_plasmidial,
    n_class_C_ampc_plasmidial, n_class_D_oxa_plasmidial,
    n_esbl_plasmidial, n_genes_efflux_plasmidial,
    has_mcr_colistin_plasmidial, has_aminoglycoside_panres_plasmidial,
], axis=1).head()

,n_class_A_carbapenemase_plasmidial,n_class_B_mbl_plasmidial,n_class_C_ampc_plasmidial,n_class_D_oxa_plasmidial,n_esbl_plasmidial,n_genes_efflux_plasmidial,has_mcr_colistin_plasmidial,has_aminoglycoside_panres_plasmidial
sample,,,,,,,,
GCA_000216055.2,0,0,0,0,0,0,0,0
GCA_000316425.1,0,0,0,0,0,2,0,0
GCA_000223095.2,0,0,0,0,0,0,0,0
GCA_036761135.1,0,0,0,1,0,2,0,0
GCA_003670255.1,0,0,0,2,0,2,0,0


In [27]:
mask_carbapenemase_BD = r["ambler_class"].isin(["B","B1","B2","B3","D"]) & r["carbapenemase_confirmada"]

In [54]:
n_carbapenemase_BD_cromossomal = n_genes_por_condicao(r, "chromosome", mask_carbapenemase_BD)
n_carbapenemase_BD_plasmidial = n_genes_por_condicao(r, "plasmid", mask_carbapenemase_BD)

total_BD = n_carbapenemase_BD_cromossomal + n_carbapenemase_BD_plasmidial
prop_carbapenemase_BD_plasmidial = (n_carbapenemase_BD_plasmidial / total_BD).fillna(0)  # 0/0 -> sem carbapenemase B/D nenhuma, 0 é a leitura certa

In [56]:
prop_carbapenemase_BD_plasmidial.name = "prop_carbapenemase_BD_plasmidial"

prop_carbapenemase_BD_plasmidial

sample
GCA_000216055.2    0.0
GCA_000316425.1    0.0
GCA_000223095.2    0.0
GCA_036761135.1    0.5
GCA_003670255.1    1.0
                  ... 
GCA_000229165.2    0.0
GCA_000229185.2    0.0
GCA_000229205.2    0.0
GCA_000229225.2    0.0
GCA_000229245.2    0.0
Name: prop_carbapenemase_BD_plasmidial, Length: 7038, dtype: float64

### Tipo de contig

In [28]:
contagem_tipo_contig = pd.crosstab(dataset.contigs["sample_id"], dataset.contigs["contig_type"])
contagem_tipo_contig.head()

contig_type,chromosome,plasmid,virus
sample_id,,,
GCA_000172875.1,293,34,13
GCA_000188755.2,116,36,47
GCA_000188815.2,46,5,12
GCA_000188875.2,43,16,14
GCA_000208825.1,139,10,0


In [29]:
percentual_contig = contagem_tipo_contig.div(contagem_tipo_contig.sum(axis=1), axis=0) * 100
percentual_contig = percentual_contig.reindex(metadata.index, fill_value=0)
percentual_contig.head()

contig_type,chromosome,plasmid,virus
sample,,,
GCA_000216055.2,97.350993,1.655629,0.993377
GCA_000316425.1,67.567568,5.675676,26.756757
GCA_000223095.2,93.650794,6.349206,0.000000
GCA_036761135.1,79.807692,18.269231,1.923077
GCA_003670255.1,74.688797,22.406639,2.904564


In [30]:
pct_contigs_plasmidial = percentual_contig.get("plasmid", 0).rename("pct_contigs_plasmidial")
pct_contigs_plasmidial.head()

sample
GCA_000216055.2     1.655629
GCA_000316425.1     5.675676
GCA_000223095.2     6.349206
GCA_036761135.1    18.269231
GCA_003670255.1    22.406639
Name: pct_contigs_plasmidial, dtype: float64

### Contigs plasmidiais: % com resistência, % com resistência + conjugação, nº com conjugação

In [31]:
contigs_plasmid = dataset.contigs[dataset.contigs["contig_type"] == "plasmid"].copy()
contigs_plasmid.head(n=3)

,seq_name,c_score,p_score,v_score,sample_id,contig_type,length_nt,topology,n_genes,n_hallmarks,...,plasmid_genes,n_plasmid_genes,virus_genes,n_virus_genes,conjugation_genes,n_conjugation_genes,genomad_amr_genes,n_genomad_amr_genes,amr_list,amr
9,ABSC01000331.1,0.0081,0.9805,0.0114,GCA_000172875.1,plasmid,3122.0,No terminal repeats,4.0,0.0,...,"[ABSC01000331.1_1, ABSC01000331.1_2, ABSC01000...",4,[],0,[],0,[],0,[],0
16,ABSC01000324.1,0.1473,0.8130,0.0397,GCA_000172875.1,plasmid,10961.0,No terminal repeats,13.0,0.0,...,"[ABSC01000324.1_1, ABSC01000324.1_10, ABSC0100...",13,[],0,[],0,[],0,[],0
20,ABSC01000320.1,0.0010,0.9962,0.0028,GCA_000172875.1,plasmid,4674.0,No terminal repeats,9.0,1.0,...,"[ABSC01000320.1_1, ABSC01000320.1_2, ABSC01000...",9,[],0,[ABSC01000320.1_7],1,[],0,[],0


In [32]:
contigs_plasmid["tem_amr"] = contigs_plasmid["amr"] > 0
contigs_plasmid["tem_conjugacao"] = contigs_plasmid["n_conjugation_genes"] > 0
contigs_plasmid["conjugacao_e_amr"] = contigs_plasmid["tem_amr"] & contigs_plasmid["tem_conjugacao"]

In [33]:
resumo_plasmid_contig = contigs_plasmid.groupby("sample_id").agg(
    n_contigs_plasmidiais=("seq_name", "count"),
    n_contigs_plasmid_amr=("tem_amr", "sum"),
    n_contigs_plasmid_conj_amr=("conjugacao_e_amr", "sum"),
).reindex(metadata.index, fill_value=0)

resumo_plasmid_contig.head()

,n_contigs_plasmidiais,n_contigs_plasmid_amr,n_contigs_plasmid_conj_amr
sample,,,
GCA_000216055.2,5,0,0
GCA_000316425.1,21,1,0
GCA_000223095.2,4,0,0
GCA_036761135.1,38,4,1
GCA_003670255.1,54,5,1


In [34]:
pct_plasmidial_amr = (
    100 * resumo_plasmid_contig["n_contigs_plasmid_amr"] / resumo_plasmid_contig["n_contigs_plasmidiais"]
).fillna(0).rename("pct_plasmidial_amr")

pct_plasmidial_conjugacao_amr = (
    100 * resumo_plasmid_contig["n_contigs_plasmid_conj_amr"] / resumo_plasmid_contig["n_contigs_plasmidiais"]
).fillna(0).rename("pct_plasmidial_conjugacao_amr")

pct_plasmidial_conjugacao_amr.head()

sample
GCA_000216055.2    0.000000
GCA_000316425.1    0.000000
GCA_000223095.2    0.000000
GCA_036761135.1    2.631579
GCA_003670255.1    1.851852
Name: pct_plasmidial_conjugacao_amr, dtype: float64

### Mobiloma -- nº de plasmídeos com maquinaria de conjugação

Substitui a flag binária original (`has_t4ss_conjugation_machinery`) por uma contagem de
CONTIGS plasmidiais com genes de conjugação (geNomad), não mais um genoma inteiro marcado
0/1 -- diferencia genoma com 1 plasmídeo conjugativo de genoma com vários.

In [35]:
n_plasmids_com_conjugacao = (
    dataset.contigs[
        (dataset.contigs["contig_type"] == "plasmid") & (dataset.contigs["n_conjugation_genes"] > 0)
    ]
    .groupby("sample_id").size()
    .reindex(metadata.index, fill_value=0).rename("n_plasmids_com_conjugacao")
)

n_plasmids_com_conjugacao.head()

sample
GCA_000216055.2    0
GCA_000316425.1    5
GCA_000223095.2    0
GCA_036761135.1    2
GCA_003670255.1    9
Name: n_plasmids_com_conjugacao, dtype: int64

### Matriz de genes por drug class plasmidial + proporção (sem as classes desbalanceadas)

In [36]:
matriz_drugclass_plasmidial = argos.groupby_genes(
    matriz_plasmidial, dataset.annotations, by="drug_class_compacted", agg="sum"
)
matriz_drugclass_plasmidial.columns = [
    f"n_plasmidial_{c.lower().replace('/', '_').replace(' ', '_')}" for c in matriz_drugclass_plasmidial.columns
]

matriz_drugclass_plasmidial.head()

,n_plasmidial_aminoglycoside,n_plasmidial_antimycobacterial,n_plasmidial_beta-lactam,n_plasmidial_biocide_antiseptic,n_plasmidial_fluoroquinolone,n_plasmidial_glycopeptide,n_plasmidial_mls,n_plasmidial_nitroimidazole,n_plasmidial_other,n_plasmidial_oxazolidinone,n_plasmidial_peptide,n_plasmidial_phenicol,n_plasmidial_phosphonic_acid,n_plasmidial_pleuromutilin,n_plasmidial_rifamycin,n_plasmidial_sulfonamide_diaminopyrimidine,n_plasmidial_tetracycline,n_plasmidial_moenomycin_antibiotic,n_plasmidial_mupirocin-like_antibiotic,n_plasmidial_thiosemicarbazone_antibiotic
sample,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
GCA_000316425.1,0,0,1,1,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0
GCA_000223095.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
GCA_036761135.1,0,0,1,1,0,0,2,0,0,0,0,0,0,0,0,1,1,0,0,0
GCA_003670255.1,1,0,3,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0


In [37]:
pct_zero_plasmidial = (matriz_drugclass_plasmidial == 0).mean()
classes_imbalanceadas = pct_zero_plasmidial[pct_zero_plasmidial > 0.80].index.tolist()

print(f"{len(classes_imbalanceadas)} classes removidas por >80% de zero:")
print(classes_imbalanceadas)

11 classes removidas por >80% de zero:
['n_plasmidial_antimycobacterial', 'n_plasmidial_glycopeptide', 'n_plasmidial_nitroimidazole', 'n_plasmidial_other', 'n_plasmidial_oxazolidinone', 'n_plasmidial_peptide', 'n_plasmidial_phosphonic_acid', 'n_plasmidial_pleuromutilin', 'n_plasmidial_moenomycin_antibiotic', 'n_plasmidial_mupirocin-like_antibiotic', 'n_plasmidial_thiosemicarbazone_antibiotic']


In [38]:
matriz_drugclass_plasmidial_filtrada = matriz_drugclass_plasmidial.drop(columns=classes_imbalanceadas)
matriz_drugclass_plasmidial_filtrada.head()

,n_plasmidial_aminoglycoside,n_plasmidial_beta-lactam,n_plasmidial_biocide_antiseptic,n_plasmidial_fluoroquinolone,n_plasmidial_mls,n_plasmidial_phenicol,n_plasmidial_rifamycin,n_plasmidial_sulfonamide_diaminopyrimidine,n_plasmidial_tetracycline
sample,,,,,,,,,
GCA_000216055.2,0,0,0,0,0,0,0,0,0
GCA_000316425.1,0,1,1,1,0,1,1,0,1
GCA_000223095.2,0,0,0,0,0,0,0,0,0
GCA_036761135.1,0,1,1,0,2,0,0,1,1
GCA_003670255.1,1,3,1,0,0,1,0,1,0


In [39]:
prop_plasmidial = matriz_drugclass_plasmidial_filtrada.div(n_genes_plasmidial, axis=0)
prop_plasmidial = prop_plasmidial.replace([np.inf, -np.inf], np.nan)
prop_plasmidial.columns = [f"prop_{c}" for c in prop_plasmidial.columns]

prop_plasmidial.head()

,prop_n_plasmidial_aminoglycoside,prop_n_plasmidial_beta-lactam,prop_n_plasmidial_biocide_antiseptic,prop_n_plasmidial_fluoroquinolone,prop_n_plasmidial_mls,prop_n_plasmidial_phenicol,prop_n_plasmidial_rifamycin,prop_n_plasmidial_sulfonamide_diaminopyrimidine,prop_n_plasmidial_tetracycline
sample,,,,,,,,,
GCA_000216055.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_000316425.1,0.000000,1.000000,1.000000,1.0,0.000000,1.000000,1.0,0.000000,1.000000
GCA_000223095.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_036761135.1,0.000000,0.166667,0.166667,0.0,0.333333,0.000000,0.0,0.166667,0.166667
GCA_003670255.1,0.142857,0.428571,0.142857,0.0,0.000000,0.142857,0.0,0.142857,0.000000


### Virulência (VFDB)

In [40]:
def matriz_vfdb_por_contig_type(vfdb, contig_type_map, contig_type):
    hits = vfdb[vfdb["SEQUENCE"].map(contig_type_map) == contig_type]
    matriz = pd.crosstab(hits["sample_id"], hits["categoria"])
    return matriz.reindex(metadata.index, fill_value=0)

contig_type_map = dataset.contigs.set_index("seq_name")["contig_type"]

vfdb_cromossomal = matriz_vfdb_por_contig_type(vfdb, contig_type_map, "chromosome")
virulence_score_cromossomal = vfdb_cromossomal.clip(upper=1).sum(axis=1).rename("virulence_score_cromossomal")

virulence_score_cromossomal.head()

sample
GCA_000216055.2    0
GCA_000316425.1    8
GCA_000223095.2    6
GCA_036761135.1    4
GCA_003670255.1    4
Name: virulence_score_cromossomal, dtype: int64

### Provírus normalizado por Mb

In [41]:
provirus_genes = dataset.genomad_genes[
    dataset.genomad_genes["gene_id"].str.contains(r"\|provirus_\d+_")
].copy()

n_proviruses = (
    provirus_genes.groupby("sample_id")["seq_name"].nunique()
    .reindex(metadata.index, fill_value=0)
)

n_proviruses.head()

sample
GCA_000216055.2    4
GCA_000316425.1    7
GCA_000223095.2    3
GCA_036761135.1    1
GCA_003670255.1    4
Name: seq_name, dtype: int64

In [42]:
n_proviruses_por_mb = (n_proviruses / genoma_mb_cromo).rename("n_proviruses_por_mb")
n_proviruses_por_mb.head()

sample
GCA_000216055.2    inf
GCA_000316425.1    inf
GCA_000223095.2    inf
GCA_036761135.1    inf
GCA_003670255.1    inf
Name: n_proviruses_por_mb, dtype: float64

## Fazendo a tabela final

In [57]:
raw_features = pd.concat([
    # resistoma -- geral
    mdr_score_cromossomal,

    # resistoma -- classe de Ambler, ESBL, efluxo, co-resistências críticas
    n_class_A_carbapenemase_cromossomal,
    n_class_A_carbapenemase_plasmidial,
    n_class_B_mbl_cromossomal,
    n_class_B_mbl_plasmidial,
    n_class_C_ampc_cromossomal,
    n_class_C_ampc_plasmidial,
    n_class_D_oxa_cromossomal,
    n_class_D_oxa_plasmidial,
    n_esbl_cromossomal,
    n_esbl_plasmidial,
    n_genes_efflux_cromossomal,
    n_genes_efflux_plasmidial,
    has_mcr_colistin_cromossomal,
    has_mcr_colistin_plasmidial,
    has_aminoglycoside_panres_cromossomal,
    has_aminoglycoside_panres_plasmidial,
    prop_carbapenemase_BD_plasmidial,


    # resistoma -- composição plasmidial por drug class
    pct_contigs_plasmidial,

    # estrutura / mobilidade
    n_plasmids_com_conjugacao,

    # composição
    gc_wide["gc_diff_plasmid_cromossomo"],

    # virulência
    virulence_score_cromossomal,
], axis=1)

# corrige NaN pra genoma sem plasmídeo (gc_diff não pode ser numérico ali)
sem_plasmidio = raw_features["pct_contigs_plasmidial"] == 0
raw_features.loc[sem_plasmidio, "gc_diff_plasmid_cromossomo"] = np.nan

In [58]:
print(raw_features.shape)
raw_features.head()

(7038, 22)


,mdr_score_cromossomal,n_class_A_carbapenemase_cromossomal,n_class_A_carbapenemase_plasmidial,n_class_B_mbl_cromossomal,n_class_B_mbl_plasmidial,n_class_C_ampc_cromossomal,n_class_C_ampc_plasmidial,n_class_D_oxa_cromossomal,n_class_D_oxa_plasmidial,n_esbl_cromossomal,...,n_genes_efflux_plasmidial,has_mcr_colistin_cromossomal,has_mcr_colistin_plasmidial,has_aminoglycoside_panres_cromossomal,has_aminoglycoside_panres_plasmidial,prop_carbapenemase_BD_plasmidial,pct_contigs_plasmidial,n_plasmids_com_conjugacao,gc_diff_plasmid_cromossomo,virulence_score_cromossomal
sample,,,,,,,,,,,,,,,,,,,,,
GCA_000216055.2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.0,1.655629,0,0.18,0
GCA_000316425.1,13,0,0,0,0,1,0,0,0,0,...,2,0,0,0,0,0.0,5.675676,5,-2.92,8
GCA_000223095.2,7,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0.0,6.349206,0,-3.05,6
GCA_036761135.1,10,0,0,0,0,0,0,1,1,0,...,2,0,0,0,0,0.5,18.269231,2,0.31,4
GCA_003670255.1,8,0,0,0,0,0,0,0,2,0,...,2,0,0,0,0,1.0,22.406639,9,0.90,4


In [59]:
raw_features.to_csv('../data/raw/raw_features.csv')